In [1]:
import kagglehub

kagglehub.login()

In [2]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('m5-forecasting-accuracy')

print("Path to competition files:", path)

import os
for dirname, _, filenames in os.walk('/kaggle/input/competitions/m5-forecasting-accuracy'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        

Path to competition files: /kaggle/input/competitions/m5-forecasting-accuracy
/kaggle/input/competitions/m5-forecasting-accuracy/calendar.csv
/kaggle/input/competitions/m5-forecasting-accuracy/sample_submission.csv
/kaggle/input/competitions/m5-forecasting-accuracy/sell_prices.csv
/kaggle/input/competitions/m5-forecasting-accuracy/sales_train_validation.csv
/kaggle/input/competitions/m5-forecasting-accuracy/sales_train_evaluation.csv


### 1. Load Dataset

In [3]:
import pandas as pd
calendar_df = pd.read_csv("/kaggle/input/competitions/m5-forecasting-accuracy/sales_train_validation.csv")
sell_prices_df = pd.read_csv("/kaggle/input/competitions/m5-forecasting-accuracy/sell_prices.csv")
sales_train_evaluation_df = pd.read_csv("/kaggle/input/competitions/m5-forecasting-accuracy/sales_train_evaluation.csv")
sales_train_validation_df = pd.read_csv("/kaggle/input/competitions/m5-forecasting-accuracy/sales_train_validation.csv")

In [4]:
calendar_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1919 entries, id to d_1913
dtypes: int64(1913), object(6)
memory usage: 446.4+ MB


In [5]:
sell_prices_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6841121 entries, 0 to 6841120
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   store_id    object 
 1   item_id     object 
 2   wm_yr_wk    int64  
 3   sell_price  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 208.8+ MB


In [6]:
sales_train_evaluation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1947 entries, id to d_1941
dtypes: int64(1941), object(6)
memory usage: 452.9+ MB


In [7]:
sales_train_validation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1919 entries, id to d_1913
dtypes: int64(1913), object(6)
memory usage: 446.4+ MB


### **Review of Dataset**
- Reduced the file size useing Downcast of datatypes

In [8]:
import numpy as np
import pandas as pd


def downcast_dtypes(df):
    """Reduces memory usage of a pandas DataFrame by downcasting numeric types.

    Converts float64/float32 to float16 and int64/int32 to int16 (or int8 if
    possible).
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print(f"Original Memory Usage: {start_mem:.2f} MB")

    for col in df.columns:
        col_type = df[col].dtype

        # Handle Integer types
        if np.issubdtype(col_type, np.integer):
            # Find the minimum and maximum value in the column
            c_min = df[col].min()
            c_max = df[col].max()

            # Downcast to int8 or int16 based on value ranges
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif (
                c_min > np.iinfo(np.int16).min
                and c_max < np.iinfo(np.int16).max
            ):
                df[col] = df[col].astype(np.int16)
            elif (
                c_min > np.iinfo(np.int32).min
                and c_max < np.iinfo(np.int32).max
            ):
                df[col] = df[col].astype(np.int32)

        # Handle Float types
        elif np.issubdtype(col_type, np.floating):
            c_min = df[col].min()
            c_max = df[col].max()

            # Downcast to float16 if it fits within the boundaries
            if (
                c_min > np.finfo(np.float16).min
                and c_max < np.finfo(np.float16).max
            ):
                df[col] = df[col].astype(np.float16)
            else:
                df[col] = df[col].astype(np.float32)

    end_mem = df.memory_usage().sum() / 1024**2
    print(f"Optimized Memory Usage: {end_mem:.2f} MB")
    print(
        f"Decreased by: {100 * (start_mem - end_mem) / start_mem:.1f}%"
    )

    return df


In [9]:
calendar_df = downcast_dtypes(calendar_df)
sell_prices_df = downcast_dtypes(sell_prices_df)
sales_train_evaluation_df = downcast_dtypes(sales_train_evaluation_df)
sales_train_validation_df = downcast_dtypes(sales_train_validation_df)

Original Memory Usage: 446.40 MB
Optimized Memory Usage: 95.00 MB
Decreased by: 78.7%
Original Memory Usage: 208.77 MB
Optimized Memory Usage: 130.48 MB
Decreased by: 37.5%
Original Memory Usage: 452.91 MB
Optimized Memory Usage: 96.13 MB
Decreased by: 78.8%
Original Memory Usage: 446.40 MB
Optimized Memory Usage: 95.00 MB
Decreased by: 78.7%


In [10]:
sales_train_validation_df.shape,sales_train_validation_df.shape

((30490, 1919), (30490, 1919))

In [11]:
submission=pd.read_csv("/kaggle/input/competitions/m5-forecasting-accuracy/sample_submission.csv")

In [12]:
submission

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60975,FOODS_3_823_WI_3_evaluation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
60976,FOODS_3_824_WI_3_evaluation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
60977,FOODS_3_825_WI_3_evaluation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
60978,FOODS_3_826_WI_3_evaluation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


###  ObjectiveGoal: Generate point forecasts for daily unit sales for 28 days into the future.
- Target Variables: 28 individual columns named F1 to F28 representing daily sales volumes for each item-store combination.

## 1. LSTM model for forcasting

#### a. Creating Dataset of LSTM

In [13]:
train_lstm=sales_train_evaluation_df

In [14]:
train_lstm.sample(n=5)

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
10598,HOUSEHOLD_2_356_CA_4_evaluation,HOUSEHOLD_2_356,HOUSEHOLD_2,HOUSEHOLD,CA_4,CA,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
27121,FOODS_3_505_WI_2_evaluation,FOODS_3_505,FOODS_3,FOODS,WI_2,WI,0,0,0,0,...,1,1,2,1,1,3,3,2,1,0
882,HOUSEHOLD_1_324_CA_1_evaluation,HOUSEHOLD_1_324,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,0,0,1,0,...,1,2,1,2,1,0,0,1,2,2
2750,FOODS_3_526_CA_1_evaluation,FOODS_3_526,FOODS_3,FOODS,CA_1,CA,4,4,6,5,...,3,4,5,4,5,4,1,3,1,4
30460,FOODS_3_798_WI_3_evaluation,FOODS_3_798,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,1,0,0,1,0,0,0,2,0,0


In [15]:
train_lstm=train_lstm.T

In [16]:
train_lstm

,0,1,2,3,4,5,6,7,8,9,...,30480,30481,30482,30483,30484,30485,30486,30487,30488,30489
id,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_006_CA_1_evaluation,HOBBIES_1_007_CA_1_evaluation,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_009_CA_1_evaluation,HOBBIES_1_010_CA_1_evaluation,...,FOODS_3_818_WI_3_evaluation,FOODS_3_819_WI_3_evaluation,FOODS_3_820_WI_3_evaluation,FOODS_3_821_WI_3_evaluation,FOODS_3_822_WI_3_evaluation,FOODS_3_823_WI_3_evaluation,FOODS_3_824_WI_3_evaluation,FOODS_3_825_WI_3_evaluation,FOODS_3_826_WI_3_evaluation,FOODS_3_827_WI_3_evaluation
item_id,HOBBIES_1_001,HOBBIES_1_002,HOBBIES_1_003,HOBBIES_1_004,HOBBIES_1_005,HOBBIES_1_006,HOBBIES_1_007,HOBBIES_1_008,HOBBIES_1_009,HOBBIES_1_010,...,FOODS_3_818,FOODS_3_819,FOODS_3_820,FOODS_3_821,FOODS_3_822,FOODS_3_823,FOODS_3_824,FOODS_3_825,FOODS_3_826,FOODS_3_827
dept_id,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,HOBBIES_1,...,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3,FOODS_3
cat_id,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,HOBBIES,...,FOODS,FOODS,FOODS,FOODS,FOODS,FOODS,FOODS,FOODS,FOODS,FOODS
store_id,CA_1,CA_1,CA_1,CA_1,CA_1,CA_1,CA_1,CA_1,CA_1,CA_1,...,WI_3,WI_3,WI_3,WI_3,WI_3,WI_3,WI_3,WI_3,WI_3,WI_3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
d_1937,0,0,0,1,0,0,1,5,0,1,...,3,6,3,0,0,1,0,1,0,0
d_1938,3,0,2,3,0,0,0,4,0,1,...,1,4,3,1,2,0,1,0,1,2
d_1939,3,0,3,0,2,5,1,1,0,0,...,3,4,3,1,1,0,0,1,1,2
d_1940,0,0,0,2,1,2,1,40,1,0,...,0,1,0,0,3,1,1,0,1,5


In [17]:
train_lstm_df=train_lstm[6:]

In [18]:
train_lstm_df

,0,1,2,3,4,5,6,7,8,9,...,30480,30481,30482,30483,30484,30485,30486,30487,30488,30489
d_1,0,0,0,0,0,0,0,12,2,0,...,0,14,1,0,4,0,0,0,0,0
d_2,0,0,0,0,0,0,0,15,0,0,...,0,11,1,0,4,0,0,6,0,0
d_3,0,0,0,0,0,0,0,0,7,1,...,0,5,1,0,2,2,0,0,0,0
d_4,0,0,0,0,0,0,0,0,3,0,...,0,6,1,0,5,2,0,2,0,0
d_5,0,0,0,0,0,0,0,0,0,0,...,0,5,1,0,2,0,0,2,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
d_1937,0,0,0,1,0,0,1,5,0,1,...,3,6,3,0,0,1,0,1,0,0
d_1938,3,0,2,3,0,0,0,4,0,1,...,1,4,3,1,2,0,1,0,1,2
d_1939,3,0,3,0,2,5,1,1,0,0,...,3,4,3,1,1,0,0,1,1,2
d_1940,0,0,0,2,1,2,1,40,1,0,...,0,1,0,0,3,1,1,0,1,5


- For unifrom Distribution make all dataset in between 0-1

In [19]:
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler(feature_range=(0,1))
train_lstm_df_scaled=scaler.fit_transform(train_lstm_df)

In [20]:
train_lstm_df_scaled

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.3       , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.6       , 0.        , 0.5       , ..., 0.05      , 0.08333333,
        0.16666667],
       [0.        , 0.        , 0.        , ..., 0.        , 0.08333333,
        0.41666667],
       [0.2       , 0.        , 0.16666667, ..., 0.1       , 0.        ,
        0.08333333]], shape=(1941, 30490))

- Creating dataset like that 14 days values we geting to predict 15 day sales
- d_1--d_14-->d_15,d_2--d_15-->d_16,...

In [21]:
timestaps=14
x_train=[]
y_train=[]
for i in range(timestaps,train_lstm_df_scaled.shape[0]):
    x_train.append(train_lstm_df_scaled[i-timestaps:i])
    y_train.append(train_lstm_df_scaled[i])

In [22]:
x_train=np.array(x_train)
y_train=np.array(y_train)

In [23]:
x_train.shape,y_train.shape

((1927, 14, 30490), (1927, 30490))

#### b. Train LSTM Model

In [24]:
from tensorflow import keras
from tensorflow.keras import layers

model=keras.Sequential()
model.add(layers.LSTM(64,activation='relu',input_shape=(x_train.shape[1],x_train.shape[2])))
model.add(layers.Dense(32,activation='relu'))
model.add(layers.Dense(30490))

model.compile(loss='mse',optimizer='adam')
model.summary()


2026-06-08 17:49:06.109775: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780940946.300906      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780940946.355688      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780940946.795832      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780940946.795868      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780940946.795871      23 computation_placer.cc:177] computation placer alr

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │     7,822,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 30490)          │     1,006,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,830,330 (33.69 MB)

 Trainable params: 8,830,330 (33.69 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
model.fit(x_train,y_train,epochs=25,batch_size=16)

Epoch 1/25


I0000 00:00:1780940971.136317      67 service.cc:152] XLA service 0x7a7140003750 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780940971.136409      67 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1780940971.549000      67 cuda_dnn.cc:529] Loaded cuDNN version 91002


  7/121 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0218

I0000 00:00:1780940972.755490      67 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


121/121 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - loss: 0.0158
Epoch 2/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0135
Epoch 3/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0132
Epoch 4/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0130
Epoch 5/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0128
Epoch 6/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0127
Epoch 7/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0126
Epoch 8/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0125
Epoch 9/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 0.0124
Epoch 10/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0123
Epoch 11/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 0.0123
Epoch 12/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0123
Epoch 13/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 0.0123
Epoch 14/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0122
Epoch 15/25
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step 

In [26]:
# Save the model to the Kaggle working directory
model.save('m5_lstm_model.keras')
print("Model saved successfully in /kaggle/working/m5_lstm_model.keras")

Model saved successfully in /kaggle/working/m5_lstm_model.keras


#### c. test model

In [27]:
from tensorflow import keras
loaded_model = keras.models.load_model('m5_lstm_model.keras')

In [28]:
inputs=train_lstm_df_scaled[-timestaps:]
inputs=scaler.transform(inputs)

In [29]:
x_test=[]
x_test.append(inputs[0:timestaps])
x_test=np.array(x_test)

In [30]:
predictions=loaded_model.predict(x_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 528ms/step


In [31]:
len(predictions)

1

In [32]:
import numpy as np
import pandas as pd

# 1. Grab the LAST 14 days of historical SCALED data (Do NOT re-scale it)
# shape will be (14, 30490)
current_batch = train_lstm_df_scaled[-timestaps:]

# We will store our 28 days of predictions here
future_predictions = []

# 2. Rollout loop for 28 days
for i in range(28):
    # Reshape current_batch to match model input: (1, 14, 30490)
    current_input = np.expand_dims(current_batch, axis=0)
    
    # Predict the next day's sales (shape: 1, 30490)
    pred = loaded_model.predict(current_input, verbose=0)
    
    # Store the prediction
    future_predictions.append(pred[0])
    
    # Update the batch: append the new prediction, slide out the oldest day
    # pred shape is (1, 30490), so we use axis=0 to stack timesteps
    current_batch = np.append(current_batch[1:], pred, axis=0)

# Convert list to numpy array (shape: 28, 30490)
future_predictions = np.array(future_predictions)

# 3. Inverse transform back to original scale
# The scaler expects a transpose or matching dimensions, so we inverse transform day-by-day or all at once
future_predictions_scaled_back = scaler.inverse_transform(future_predictions)

# Transpose to get shapes of (30490 rows/items, 28 columns/days)
final_forecasts = future_predictions_scaled_back.T
print("Forecast shape (should be 30490, 28):", final_forecasts.shape)

Forecast shape (should be 30490, 28): (30490, 28)


In [33]:
# Load sample submission
submission = pd.read_csv("/kaggle/input/competitions/m5-forecasting-accuracy/sample_submission.csv")

# Create column names F1 to F28
forecast_cols = [f"F{i}" for i in range(1, 29)]

# Split submission into validation and evaluation blocks
sub_validation = submission[submission['id'].str.endswith('_validation')].copy()
sub_evaluation = submission[submission['id'].str.endswith('_evaluation')].copy()

# Map your predictions into the evaluation block 
# (Assuming your final_forecasts ordering matches the row order of evaluation data)
sub_evaluation[forecast_cols] = final_forecasts

# Since we don't have separate validation forecasts here, 
# you can either put zeros or match them if needed. 
# For true evaluation track submission, sub_validation can safely be left as 0s or placeholder.
sub_validation[forecast_cols] = 0 

# Combine them back together
final_submission = pd.concat([sub_validation, sub_evaluation], axis=0).reset_index(drop=True)

# Verify everything looks right
print(final_submission.head())
print(final_submission.tail())

# Save to csv
final_submission.to_csv("submission.csv", index=False)
print("Submission file successfully saved as submission.csv!")

                              id   F1   F2   F3   F4   F5   F6   F7   F8   F9  \
0  HOBBIES_1_001_CA_1_validation  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
1  HOBBIES_1_002_CA_1_validation  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
2  HOBBIES_1_003_CA_1_validation  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
3  HOBBIES_1_004_CA_1_validation  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
4  HOBBIES_1_005_CA_1_validation  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   

   ...  F19  F20  F21  F22  F23  F24  F25  F26  F27  F28  
0  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
1  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
2  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
3  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
4  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  

[5 rows x 29 columns]
                                id        F1        F2        F3        F4  \
60975  FOODS_3_823_WI_3_evaluation  0.622850  0.640569  0.